# Offensive AI API Security — Resumed 30,000-Record Training Experiment

This notebook executes the exact resumption of the 30,000-record classification training:
- Resumption Checkpoint: `checkpoint-1250` (Step 1,250 of 1,875, loss=1.816e-5, epoch=0.6667)
- Remaining Workload: Exactly 625 optimizer steps (Steps 1,251 to 1,875, ~10,000 records)
- Hardware: 2x Tesla T4 GPUs via PyTorch DDP (`torchrun --nproc_per_node=2`)
- Preserved Optimizer & Scheduler: Restored exactly from AdamW `optimizer.pt` and cosine `scheduler.pt`
- Checkpoint Interval: Every 125 steps (1375, 1500, 1625, 1750, 1875)
- Immediate Save: `classification_training_checkpoint_final` saved immediately before evaluations
- Evaluation Benchmarks:
  1. Original Validation Benchmark (1,000 records)
  2. Corrected Validation Benchmark (1,000 records)
  3. XSS/RCE Boundary Contrast Diagnostic Subset (700 records)
  4. Quarantined Independent Test Benchmark (`master_test_4100_repaired.jsonl`)

In [ ]:
# 1. Environment & Dependencies Setup
!nvidia-smi
!pip install -q bitsandbytes accelerate peft transformers datasets psutil scikit-learn


In [ ]:
# 2. Verify Mounted Kaggle Inputs
import os
from pathlib import Path

print('Listing /kaggle/input structure:')
input_root = Path('/kaggle/input')
if input_root.exists():
    for d in sorted(input_root.iterdir()):
        print(f' * {d.name}/')
        if d.is_dir():
            for f in sorted(list(d.glob('**/*')))[:15]:
                if f.is_file():
                    print(f'     - {f.relative_to(d)} ({f.stat().st_size:,} bytes)')
else:
    print('Warning: /kaggle/input does not exist!')


In [ ]:
# 3. Write Resumption Training & Evaluation Script
from pathlib import Path

script_file = Path('/kaggle/working/resume_expanded_classification.py')
script_file.write_text("\"\"\"\nresume_expanded_classification.py - Resumed 2x Tesla T4 GPU Training from Checkpoint-1250 (Full 30,000 Records)\nConfiguration:\n1. Exact Resumption Point: Checkpoint-1250 (Step 1250 / 1875, loss=1.816e-5, epoch=0.6667)\n2. Preserved Training Dataset: classification_training_30000.jsonl (30,000 records, SHA256: 394b43a16dba4e959e7d7b1e8f93dc22aa4f0af24a36b9db34b2b25c76f873dc)\n3. Preserved Optimizer & Scheduler: Restored exactly from optimizer.pt and scheduler.pt (continuing LR ~1.25e-5)\n4. Remaining Workload: Exactly 625 optimizer steps (Step 1251 -> 1875, remaining 10,000 samples)\n5. Hardware: 2x Tesla T4 GPUs via PyTorch DDP (torchrun --nproc_per_node=2)\n6. Checkpoint Strategy: Save every 125 steps (1375, 1500, 1625, 1750, 1875) with save_total_limit=5\n7. Immediate Adapter Persistence: final adapter saved immediately to classification_training_checkpoint_final\n8. Evaluation Scope: Original Val (1,000), Corrected Val (1,000), Boundary Subset (700), and Independent Test Set (4,100)\n\"\"\"\n\nimport os\nimport sys\nimport time\nimport json\nimport shutil\nimport psutil\nimport inspect\nfrom pathlib import Path\nfrom collections import Counter, defaultdict\nimport torch\nimport torch.distributed as dist\nfrom datasets import Dataset\nfrom peft import PeftModel, prepare_model_for_kbit_training\nfrom transformers import (\n    AutoTokenizer,\n    AutoModelForCausalLM,\n    Trainer,\n    TrainingArguments,\n    TrainerCallback,\n    BitsAndBytesConfig\n)\nimport peft.utils.save_and_load as _peft_save_load\n\ndef _ddp_skip_tensor_parallel_sharding(model, state_dict, adapter_name):\n    for module in model.modules():\n        getter = getattr(module, \"get_base_layer\", None)\n        if getter is None:\n            continue\n        base_layer = getter()\n        if (\n            getattr(base_layer, \"_hf_tp_plan\", None) is not None\n            and getattr(base_layer, \"_hf_device_mesh\", None) is not None\n        ):\n            raise RuntimeError(\"DDP workaround is only valid without tensor parallelism.\")\n    return None\n\n_peft_save_load._maybe_shard_state_dict_for_tp = _ddp_skip_tensor_parallel_sharding\n\n# Distributed Setup\nis_distributed = \"LOCAL_RANK\" in os.environ or \"RANK\" in os.environ\nif is_distributed:\n    local_rank = int(os.environ.get(\"LOCAL_RANK\", 0))\n    global_rank = int(os.environ.get(\"RANK\", 0))\n    world_size = int(os.environ.get(\"WORLD_SIZE\", 1))\n    torch.cuda.set_device(local_rank)\n    if not dist.is_initialized():\n        dist.init_process_group(backend=\"nccl\")\nelse:\n    local_rank = 0\n    global_rank = 0\n    world_size = 1\n\ndef log_rank0(msg):\n    if global_rank == 0:\n        print(f\"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}\", flush=True)\n\n# Paths Configuration\nBASE_MODEL = \"Qwen/Qwen2.5-1.5B-Instruct\"\n\ndef find_checkpoint(input_dir, root_dir):\n    if Path(\"/kaggle/working\").exists():\n        candidates = [\n            input_dir / \"offensive-ai-checkpoint-1250\" / \"checkpoint-1250\",\n            input_dir / \"offensive-ai-checkpoint-1250\",\n            input_dir / \"urop-offensive-ai\" / \"classification_training_output\" / \"checkpoint-1250\",\n            input_dir / \"urop-offensive-ai\" / \"checkpoint-1250\",\n            input_dir / \"classification-checkpoint-1250\" / \"checkpoint-1250\",\n            input_dir / \"classification-checkpoint-1250\",\n        ]\n        for c in candidates:\n            if c.exists() and (c / \"adapter_model.safetensors\").exists():\n                return c\n        for c in input_dir.glob(\"**/checkpoint-1250\"):\n            if (c / \"adapter_model.safetensors\").exists():\n                return c\n        raise FileNotFoundError(f\"Could not find checkpoint-1250 in {input_dir}\")\n    else:\n        return root_dir / \"evaluation\" / \"expanded_classification_training\" / \"classification_training_output\" / \"checkpoint-1250\"\n\ndef find_test_file(input_dir, root_dir):\n    candidates = [\n        input_dir / \"offensive-ai-quarantined-test\" / \"master_test_4100_repaired.jsonl\",\n        input_dir / \"offensive-ai-classification-training\" / \"master_test_4100_repaired.jsonl\",\n        input_dir / \"classification-checkpoint-1250\" / \"master_test_4100_repaired.jsonl\",\n        root_dir / \"master_test_4100_repaired.jsonl\",\n        root_dir / \"dataset\" / \"splits\" / \"master_test_4100_repaired.jsonl\",\n    ]\n    for c in candidates:\n        if c.exists():\n            return c\n    if input_dir.exists():\n        for c in input_dir.glob(\"**/master_test_4100_repaired.jsonl\"):\n            return c\n    return root_dir / \"master_test_4100_repaired.jsonl\"\n\nif Path(\"/kaggle/working\").exists():\n    ROOT = Path(\"/kaggle/working\")\n    INPUT_DIR = Path(\"/kaggle/input\")\n    \n    # Dataset directory\n    ds_dir = INPUT_DIR / \"offensive-ai-classification-training\"\n    \n    TRAIN_PATH = ds_dir / \"classification_training_30000.jsonl\"\n    VAL_ORIG_PATH = ds_dir / \"validation_benchmark_original_1000.jsonl\"\n    VAL_CORR_PATH = ds_dir / \"validation_benchmark_corrected_1000.jsonl\"\n    BOUNDARY_PATH = ds_dir / \"boundary_contrast_subset_700.jsonl\"\n    \n    TEST_PATH = find_test_file(INPUT_DIR, ROOT)\n    RAW_RESUME_CKPT = find_checkpoint(INPUT_DIR, ROOT)\n    OUTPUT_DIR = ROOT / \"classification_training_output\"\nelse:\n    ROOT = Path(\".\")\n    INPUT_DIR = ROOT / \"dataset\"\n    TRAIN_PATH = ROOT / \"dataset\" / \"splits\" / \"classification_training_30000.jsonl\"\n    VAL_ORIG_PATH = ROOT / \"dataset\" / \"splits\" / \"validation_benchmark_original_1000.jsonl\"\n    VAL_CORR_PATH = ROOT / \"dataset\" / \"splits\" / \"validation_benchmark_corrected_1000.jsonl\"\n    BOUNDARY_PATH = ROOT / \"dataset\" / \"splits\" / \"boundary_contrast_subset_700.jsonl\"\n    TEST_PATH = find_test_file(INPUT_DIR, ROOT)\n    RAW_RESUME_CKPT = find_checkpoint(INPUT_DIR, ROOT)\n    OUTPUT_DIR = ROOT / \"classification_training_output\"\n\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\nLOCAL_RESUME_CKPT = OUTPUT_DIR / \"checkpoint-1250\"\n\n# Stage Checkpoint-1250 to writeable directory on Rank 0\nif global_rank == 0 and not LOCAL_RESUME_CKPT.exists():\n    log_rank0(f\"Staging Checkpoint-1250 from {RAW_RESUME_CKPT} to {LOCAL_RESUME_CKPT}...\")\n    shutil.copytree(str(RAW_RESUME_CKPT), str(LOCAL_RESUME_CKPT))\n    log_rank0(\"Checkpoint-1250 staged successfully.\")\n\nif is_distributed:\n    dist.barrier()\n\nMAX_LENGTH = 1024\n\nlog_rank0(\"=\" * 80)\nlog_rank0(\"RESUMING EXPANDED CLASSIFICATION TRAINING (30,000 TOTAL RECORDS)\")\nlog_rank0(f\"Resuming From:      {LOCAL_RESUME_CKPT}\")\nlog_rank0(f\"Distributed DDP:    {is_distributed} | Local Rank: {local_rank} | World Size: {world_size}\")\nlog_rank0(f\"Base Model:         {BASE_MODEL}\")\nlog_rank0(f\"Training Dataset:   {TRAIN_PATH}\")\nlog_rank0(f\"Val Benchmark Orig: {VAL_ORIG_PATH}\")\nlog_rank0(f\"Val Benchmark Corr: {VAL_CORR_PATH}\")\nlog_rank0(f\"Boundary Dataset:   {BOUNDARY_PATH}\")\nlog_rank0(f\"Test Dataset:       {TEST_PATH}\")\nlog_rank0(\"=\" * 80)\n\n# Tokenizer\ntokenizer = AutoTokenizer.from_pretrained(\n    str(LOCAL_RESUME_CKPT) if LOCAL_RESUME_CKPT.exists() else BASE_MODEL,\n    trust_remote_code=True\n)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\ntokenizer.padding_side = \"right\"\n\n# 4-bit Quantization Config\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type=\"nf4\",\n    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,\n    bnb_4bit_use_double_quant=True,\n)\n\n# Load Training Data\nlog_rank0(\"Loading 30,000 classification training samples...\")\nwith open(TRAIN_PATH, \"r\", encoding=\"utf-8\") as f:\n    train_records = [json.loads(line) for line in f if line.strip()]\n\nlog_rank0(f\"Loaded {len(train_records)} training records. Tokenizing...\")\n\ndef format_records(records):\n    formatted = []\n    classes = [\n        \"Benign\", \"Cookie Injection\", \"Directory Traversal\", \"LOG4J\",\n        \"Log Forging\", \"RCE\", \"SQL Injection\", \"XSS\"\n    ]\n    for r in records:\n        msgs = r[\"messages\"]\n        prompt_text = tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)\n        response_text = msgs[2][\"content\"].strip() + tokenizer.eos_token\n        full_text = prompt_text + response_text\n        \n        tokenized_prompt = tokenizer(prompt_text, add_special_tokens=False)[\"input_ids\"]\n        tokenized_full = tokenizer(full_text, add_special_tokens=False)[\"input_ids\"]\n        \n        if len(tokenized_full) > MAX_LENGTH:\n            tokenized_full = tokenized_full[:MAX_LENGTH]\n            \n        labels = list(tokenized_full)\n        prompt_len = min(len(tokenized_prompt), len(labels))\n        for i in range(prompt_len):\n            labels[i] = -100\n            \n        formatted.append({\n            \"input_ids\": tokenized_full,\n            \"labels\": labels,\n            \"length\": len(tokenized_full)\n        })\n    return formatted\n\ntrain_data = format_records(train_records)\ntrain_dataset = Dataset.from_list(train_data)\nlog_rank0(f\"Tokenized {len(train_dataset)} records.\")\n\n# Dynamic Padding Collator\nclass DataCollatorForDynamicPadding:\n    def __init__(self, pad_token_id, pad_label_id=-100):\n        self.pad_token_id = pad_token_id\n        self.pad_label_id = pad_label_id\n\n    def __call__(self, features):\n        batch_lens = [len(f[\"input_ids\"]) for f in features]\n        max_batch_len = max(batch_lens)\n        \n        batch_input_ids = []\n        batch_labels = []\n        batch_attention_mask = []\n\n        for f in features:\n            ids = f[\"input_ids\"]\n            labs = f[\"labels\"]\n            pad_len = max_batch_len - len(ids)\n            \n            padded_ids = ids + [self.pad_token_id] * pad_len\n            padded_labs = labs + [self.pad_label_id] * pad_len\n            attn_mask = [1] * len(ids) + [0] * pad_len\n\n            batch_input_ids.append(padded_ids)\n            batch_labels.append(padded_labs)\n            batch_attention_mask.append(attn_mask)\n\n        return {\n            \"input_ids\": torch.tensor(batch_input_ids, dtype=torch.long),\n            \"labels\": torch.tensor(batch_labels, dtype=torch.long),\n            \"attention_mask\": torch.tensor(batch_attention_mask, dtype=torch.long),\n        }\n\ncollator = DataCollatorForDynamicPadding(tokenizer.pad_token_id)\n\n# Model Loading & Base Architecture\nlog_rank0(f\"Loading Base Model: {BASE_MODEL}...\")\ndevice_map = {\"\": local_rank} if is_distributed else ({\"\": 0} if torch.cuda.is_available() else None)\n\nbase_model = AutoModelForCausalLM.from_pretrained(\n    BASE_MODEL,\n    quantization_config=bnb_config,\n    device_map=device_map,\n    trust_remote_code=True\n)\nbase_model = prepare_model_for_kbit_training(base_model)\n\nlog_rank0(f\"Attaching adapter from Checkpoint-1250: {LOCAL_RESUME_CKPT}...\")\nmodel = PeftModel.from_pretrained(\n    base_model,\n    str(LOCAL_RESUME_CKPT),\n    is_trainable=True\n)\nmodel.print_trainable_parameters()\n\n# Training Arguments\ntrain_args_kwargs = {\n    \"output_dir\": str(OUTPUT_DIR),\n    \"learning_rate\": 5e-5,\n    \"lr_scheduler_type\": \"cosine\",\n    \"warmup_ratio\": 0.05,\n    \"num_train_epochs\": 1.0,\n    \"max_steps\": 1875,\n    \"per_device_train_batch_size\": 2,\n    \"gradient_accumulation_steps\": 4,\n    \"bf16\": torch.cuda.is_bf16_supported(),\n    \"fp16\": not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),\n    \"logging_steps\": 25,\n    \"save_strategy\": \"steps\",\n    \"save_steps\": 125,\n    \"save_total_limit\": 5,\n    \"group_by_length\": True,\n    \"length_column_name\": \"length\",\n    \"report_to\": \"none\",\n    \"gradient_checkpointing\": True,\n    \"dataloader_drop_last\": False,\n    \"ddp_find_unused_parameters\": False if is_distributed else None,\n}\n\nvalid_args = inspect.signature(TrainingArguments.__init__).parameters\nfiltered_args = {k: v for k, v in train_args_kwargs.items() if k in valid_args and v is not None}\ntraining_args = TrainingArguments(**filtered_args)\n\nlog_rank0(\"Instantiating Trainer...\")\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_dataset,\n    data_collator=collator,\n)\n\n# Resumed Training Execution\nlog_rank0(\"=\" * 80)\nlog_rank0(f\"RESUMING TRAINING FROM STEP 1250 -> 1875 (Remaining 625 optimizer steps)...\")\nlog_rank0(\"=\" * 80)\nt_start = time.time()\ntrain_result = trainer.train(resume_from_checkpoint=str(LOCAL_RESUME_CKPT))\ntrain_duration = time.time() - t_start\n\nlog_rank0(\"=\" * 80)\nlog_rank0(f\"REMAINING TRAINING COMPLETED in {train_duration:.2f}s ({train_duration/60:.2f} min)\")\nlog_rank0(f\"Final Global Step:  {train_result.global_step}\")\nlog_rank0(f\"Final Train Loss:   {train_result.training_loss:.4f}\")\nlog_rank0(\"=\" * 80)\n\n# Save Final Trained Adapter IMMEDIATELY\nfinal_adapter_dir = OUTPUT_DIR / \"classification_training_checkpoint_final\"\nif global_rank == 0:\n    log_rank0(f\"Saving final trained adapter to {final_adapter_dir}...\")\n    trainer.save_model(str(final_adapter_dir))\n    tokenizer.save_pretrained(str(final_adapter_dir))\n    log_rank0(\"Final adapter saved successfully.\")\n\n# DDP Collective Synchronization\nif is_distributed:\n    dist.barrier()\n    log_rank0(\"Collective DDP synchronization complete.\")\n\n# Post-Training Comprehensive Evaluation on Rank 0\nif global_rank == 0:\n    log_rank0(\"\\n\" + \"=\" * 80)\n    log_rank0(\"STARTING COMPREHENSIVE FINAL MODEL EVALUATION\")\n    log_rank0(\"=\" * 80)\n    \n    del model\n    del trainer\n    del base_model\n    import gc\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n    classes = [\n        \"Benign\", \"Cookie Injection\", \"Directory Traversal\", \"LOG4J\",\n        \"Log Forging\", \"RCE\", \"SQL Injection\", \"XSS\"\n    ]\n    \n    def evaluate_model_on_dataset(eval_model, dataset_path, eval_name, max_samples=None):\n        log_rank0(f\"Evaluating on {dataset_path.name} ({eval_name})...\")\n        with open(dataset_path, \"r\", encoding=\"utf-8\") as f:\n            raw_recs = [json.loads(line) for line in f if line.strip()]\n            \n        if max_samples and len(raw_recs) > max_samples:\n            log_rank0(f\"Subsampling to {max_samples} balanced samples for bounded execution...\")\n            by_cat = defaultdict(list)\n            for r in raw_recs:\n                by_cat[r.get(\"category\", \"Benign\")].append(r)\n            per_cat_limit = max(1, max_samples // len(by_cat))\n            raw_recs = [r for cat_items in by_cat.values() for r in cat_items[:per_cat_limit]]\n\n        y_true = []\n        y_pred = []\n        t0 = time.time()\n        \n        for idx, rec in enumerate(raw_recs):\n            y_true.append(rec[\"category\"])\n            prompt_msgs = [rec[\"messages\"][0], rec[\"messages\"][1]]\n            prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)\n            inputs = tokenizer(prompt_text, return_tensors=\"pt\").to(eval_model.device)\n            \n            with torch.no_grad():\n                gen_out = eval_model.generate(\n                    **inputs,\n                    max_new_tokens=15,\n                    pad_token_id=tokenizer.pad_token_id,\n                    eos_token_id=tokenizer.eos_token_id,\n                    do_sample=False\n                )\n            pred_text = tokenizer.decode(gen_out[0][inputs[\"input_ids\"].shape[1]:], skip_special_tokens=True).strip()\n            \n            matched = \"Unknown\"\n            for c in classes:\n                if c.lower() in pred_text.lower():\n                    matched = c\n                    break\n            if matched == \"Unknown\":\n                matched = pred_text\n            y_pred.append(matched)\n\n        dur = time.time() - t0\n        correct = sum(1 for yt, yp in zip(y_true, y_pred) if yt == yp)\n        acc = correct / len(y_true) if y_true else 0.0\n\n        per_class = {}\n        f1_list = []\n        weighted_f1_sum = 0\n        for c in classes:\n            tp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp == c)\n            fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt != c and yp == c)\n            fn = sum(1 for yt, yp in zip(y_true, y_pred) if yt == c and yp != c)\n            supp = sum(1 for yt in y_true if yt == c)\n\n            prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0\n            rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0\n            f1 = (2 * prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0\n            if supp > 0:\n                f1_list.append(f1)\n                weighted_f1_sum += (f1 * supp)\n\n            per_class[c] = {\n                \"precision\": round(prec, 4),\n                \"recall\": round(rec, 4),\n                \"f1_score\": round(f1, 4),\n                \"support\": supp\n            }\n\n        macro_f1 = sum(f1_list) / len(f1_list) if f1_list else 0.0\n        weighted_f1 = weighted_f1_sum / len(y_true) if y_true else 0.0\n\n        confusion_mat = {c_true: {c_pred: 0 for c_pred in classes + [\"Unknown\"]} for c_true in classes}\n        for yt, yp in zip(y_true, y_pred):\n            pred_col = yp if yp in classes else \"Unknown\"\n            if yt in confusion_mat:\n                confusion_mat[yt][pred_col] += 1\n\n        # Specific diagnostics\n        benign_fp = sum(1 for yt, yp in zip(y_true, y_pred) if yt == \"Benign\" and yp != \"Benign\")\n        benign_total = sum(1 for yt in y_true if yt == \"Benign\")\n        benign_fp_rate = benign_fp / benign_total if benign_total > 0 else 0.0\n\n        xss_to_rce = sum(1 for yt, yp in zip(y_true, y_pred) if yt == \"XSS\" and yp == \"RCE\")\n        rce_to_xss = sum(1 for yt, yp in zip(y_true, y_pred) if yt == \"RCE\" and yp == \"XSS\")\n\n        return {\n            \"total_samples\": len(y_true),\n            \"accuracy\": round(acc, 4),\n            \"macro_f1\": round(macro_f1, 4),\n            \"weighted_f1\": round(weighted_f1, 4),\n            \"per_class\": per_class,\n            \"confusion_matrix\": confusion_mat,\n            \"diagnostics\": {\n                \"benign_fp_count\": benign_fp,\n                \"benign_fp_rate\": round(benign_fp_rate, 4),\n                \"xss_classified_as_rce\": xss_to_rce,\n                \"rce_classified_as_xss\": rce_to_xss\n            },\n            \"duration_seconds\": round(dur, 2)\n        }\n\n    # Load Newly Trained Final Model\n    log_rank0(f\"Loading final trained model from {final_adapter_dir}...\")\n    b = AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL,\n        quantization_config=bnb_config,\n        device_map={\"\": 0} if torch.cuda.is_available() else None,\n        trust_remote_code=True\n    )\n    final_eval_model = PeftModel.from_pretrained(b, str(final_adapter_dir))\n    final_eval_model.eval()\n\n    eval_orig = evaluate_model_on_dataset(final_eval_model, VAL_ORIG_PATH, \"Final_Model_Original_Val\")\n    eval_corr = evaluate_model_on_dataset(final_eval_model, VAL_CORR_PATH, \"Final_Model_Corrected_Val\")\n    eval_boundary = evaluate_model_on_dataset(final_eval_model, BOUNDARY_PATH, \"Final_Model_Boundary_Subset\")\n    \n    eval_test = None\n    if TEST_PATH.exists():\n        eval_test = evaluate_model_on_dataset(final_eval_model, TEST_PATH, \"Final_Model_Independent_Test\", max_samples=1000)\n\n    # Compile Final Report\n    report = {\n        \"status\": \"COMPLETED\",\n        \"experiment\": \"RESUMED_EXPANDED_CLASSIFICATION_TRAINING_30000\",\n        \"resumed_from_checkpoint\": str(LOCAL_RESUME_CKPT),\n        \"starting_step\": 1250,\n        \"final_step\": train_result.global_step,\n        \"total_planned_steps\": 1875,\n        \"is_exact_resumption\": True,\n        \"total_training_records\": len(train_dataset),\n        \"completed_epochs\": 1.0,\n        \"train_loss\": round(train_result.training_loss, 4),\n        \"train_duration_seconds\": round(train_duration, 2),\n        \"saved_final_adapter\": str(final_adapter_dir),\n        \"benchmarks\": {\n            \"original_validation_benchmark_1000\": eval_orig,\n            \"corrected_validation_benchmark_1000\": eval_corr,\n            \"boundary_contrast_diagnostic_700\": eval_boundary,\n            \"independent_test_benchmark\": eval_test\n        }\n    }\n\n    report_path = OUTPUT_DIR / \"classification_training_metrics_report.json\"\n    with open(report_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(report, f, indent=2)\n\n    log_rank0(\"\\n\" + \"=\" * 80)\n    log_rank0(\"FINAL MODEL BENCHMARK RESULTS (ACCURACY / MACRO F1):\")\n    log_rank0(f\"1. Original Val (1,000):   Accuracy={eval_orig['accuracy']*100:.2f}% | Macro F1={eval_orig['macro_f1']:.4f} | Benign FP Rate={eval_orig['diagnostics']['benign_fp_rate']*100:.2f}%\")\n    log_rank0(f\"2. Corrected Val (1,000):  Accuracy={eval_corr['accuracy']*100:.2f}% | Macro F1={eval_corr['macro_f1']:.4f} | Benign FP Rate={eval_corr['diagnostics']['benign_fp_rate']*100:.2f}%\")\n    log_rank0(f\"3. Boundary Diagnostic (700): Accuracy={eval_boundary['accuracy']*100:.2f}% | Macro F1={eval_boundary['macro_f1']:.4f} | XSS->RCE={eval_boundary['diagnostics']['xss_classified_as_rce']}\")\n    if eval_test:\n        log_rank0(f\"4. Independent Test (1,000):  Accuracy={eval_test['accuracy']*100:.2f}% | Macro F1={eval_test['macro_f1']:.4f} | Benign FP Rate={eval_test['diagnostics']['benign_fp_rate']*100:.2f}%\")\n    log_rank0(\"=\" * 80)\n    log_rank0(f\"Report written to: {report_path}\")\n    log_rank0(\"TRAINING AND COMPREHENSIVE BENCHMARK EVALUATION FINISHED SUCCESSFULLY.\")\n", encoding='utf-8')
print(f'Wrote {script_file.name}: {script_file.stat().st_size:,} bytes')


In [ ]:
# 4. Execute Resumed Training via 2x Tesla T4 PyTorch DDP
import subprocess
import sys
import os
import torch

env = os.environ.copy()
for k in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE', 'MASTER_ADDR', 'MASTER_PORT']:
    env.pop(k, None)
env['CUDA_VISIBLE_DEVICES'] = '0,1'
env['TOKENIZERS_PARALLELISM'] = 'false'
env['OMP_NUM_THREADS'] = '4'

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'Detected available GPU count: {num_gpus}')

if num_gpus > 1:
    cmd = [
        'torchrun',
        '--standalone',
        '--nnodes=1',
        f'--nproc_per_node={num_gpus}',
        '/kaggle/working/resume_expanded_classification.py'
    ]
else:
    cmd = [sys.executable, '/kaggle/working/resume_expanded_classification.py']

print('Executing:', ' '.join(cmd))
res = subprocess.run(cmd, env=env, capture_output=False)
if res.returncode != 0:
    raise RuntimeError(f'Resumed training failed with exit code {res.returncode}')
print('Resumed training and comprehensive evaluation finished successfully!')


In [ ]:
# 5. Display Metrics Report & Artifact Files
import json
from pathlib import Path

report_path = Path('/kaggle/working/classification_training_output/classification_training_metrics_report.json')
if report_path.exists():
    print('=' * 80)
    print('FINAL CLASSIFICATION TRAINING METRICS REPORT:')
    print('=' * 80)
    print(report_path.read_text(encoding='utf-8'))
else:
    print('Warning: Metrics report not found!')

print('\nClassification training output directory contents:')
for p in sorted(Path('/kaggle/working/classification_training_output').glob('**/*')):
    if p.is_file():
        print(f'{p.relative_to("/kaggle/working")}: {p.stat().st_size:,} bytes')
